In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
from pyspark.sql.types import ArrayType, StructType
import time, os, glob
spark = SparkSession.builder \
    .appName("PySparkPractice") \
    .getOrCreate()

In [4]:
BASE = "/content/"
for f in ["shipments_75000.csv", "shipments_extra_12000.csv", "warehouse_master.csv",
          "carrier_master.csv", "route_master.csv", "warehouse_sensor_config.json"]:
    print(f, "->", os.path.exists(BASE + f))

read_csv = lambda name: spark.read.csv(BASE + name, header=True, inferSchema=True)

shipments_df = read_csv("shipments_75000.csv")
shipments_extra_df = read_csv("shipments_extra_12000.csv")
warehouse_df = read_csv("warehouse_master.csv")
carrier_df = read_csv("carrier_master.csv")
route_df = read_csv("route_master.csv")


shipments_75000.csv -> True
shipments_extra_12000.csv -> True
warehouse_master.csv -> True
carrier_master.csv -> True
route_master.csv -> True
warehouse_sensor_config.json -> True


**PART A | DataFrame Operations**

1.Display the schema, total number of records, and first 15 shipments.

In [5]:
shipments_df.printSchema()
print("Total records:", shipments_df.count())
shipments_df.show(15)

root
 |-- shipment_id: string (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- dispatch_date: date (nullable = true)
 |-- promised_delivery_date: date (nullable = true)
 |-- actual_delivery_date: date (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- carrier_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- service_level: string (nullable = true)
 |-- package_weight_kg: double (nullable = true)
 |-- shipping_fee_inr: double (nullable = true)
 |-- shipping_cost_inr: double (nullable = true)
 |-- shipment_status: string (nullable = true)
 |-- delivery_attempts: integer (nullable = true)
 |-- damage_flag: string (nullable = true)
 |-- customer_rating: double (nullable = true)
 |-- handling_minutes: double (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- priority_flag: string (nullable = true)

Total records: 75000
+-----------+------------+-------------+--------------

2.Select shipment_id, warehouse_id, service_level, shipment_status, and shipping_fee_inr.

In [6]:
shipments_df.select("shipment_id", "warehouse_id", "service_level", "shipment_status", "shipping_fee_inr").show()

+-----------+------------+-------------+---------------+----------------+
|shipment_id|warehouse_id|service_level|shipment_status|shipping_fee_inr|
+-----------+------------+-------------+---------------+----------------+
| SHP0000001|         W08|      Express|      Delivered|          554.34|
| SHP0000002|         W08|     Standard|      Delivered|          414.56|
| SHP0000003|         W01|      Express|      Delivered|          724.64|
| SHP0000004|         W03|     Standard|      Delivered|          287.86|
| SHP0000005|         W03|      Economy|      Delivered|          251.68|
| SHP0000006|         W01|      Express|      Delivered|           477.9|
| SHP0000007|         W02|     Standard|     In Transit|          298.68|
| SHP0000008|         W03|      Economy|     In Transit|          284.52|
| SHP0000009|         W02|     Standard|      Delivered|          861.74|
| SHP0000010|         W02|      Economy|      Delivered|          309.76|
| SHP0000011|         W04|     Standar

3.Filter shipments where service_level = "Express" and package_weight_kg > 5.

In [7]:
shipments_df.filter((F.col("service_level") == "Express") & (F.col("package_weight_kg") > 5)).show()

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
| SHP0000003|  2026-01-31|   2026-02-01|            2026-02-03| 

4.Sort shipments by shipping_fee_inr in descending order and display the top 20.

In [8]:
shipments_df.orderBy(F.col("shipping_fee_inr").desc()).show()

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|    payment_mode|priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
| SHP0044904|  2026-08-11|   2026-08-13|            2026-08-16| 

**PART B | Aggregations and Joins**

5.Calculate the total number of shipments for each shipment_status.

In [10]:
shipments_df.groupBy("shipment_status").count().orderBy(F.desc("count")).show()

+---------------+-----+
|shipment_status|count|
+---------------+-----+
|      Delivered|52434|
|     In Transit|10472|
|       Returned| 6130|
|      Cancelled| 3723|
|           Lost| 2241|
+---------------+-----+



6.Calculate total shipping fees, total shipping costs, and profit margin for each service_level.
Handle missing cost values appropriately.

In [13]:
shipments_df.groupBy("service_level").agg(
        F.count("*").alias("shipments"),
        F.sum(F.col("shipping_cost_inr").isNull().cast("int")).alias("missing_cost_rows"),
        F.round(F.sum("shipping_fee_inr"), 2).alias("total_fees"),
        F.round(F.sum("shipping_cost_inr"), 2).alias("total_cost"),
        F.round(F.sum(F.when(F.col("shipping_cost_inr").isNotNull(), F.col("shipping_fee_inr"))), 2).alias("fee_with_known_cost")).withColumn("profit_margin_pct",
                   F.round((F.col("fee_with_known_cost") - F.col("total_cost")) / F.col("fee_with_known_cost") * 100, 2)).show()

+-------------+---------+-----------------+-------------+------------+-------------------+-----------------+
|service_level|shipments|missing_cost_rows|   total_fees|  total_cost|fee_with_known_cost|profit_margin_pct|
+-------------+---------+-----------------+-------------+------------+-------------------+-----------------+
|      Express|    21090|              401|1.005846806E7|   7350805.7|         9865578.22|            25.49|
|     Standard|    40386|              844|1.343737262E7|1.01104659E7|      1.315425914E7|            23.14|
|      Economy|    13524|              258|   4489918.68|   3386212.5|          4404331.7|            23.12|
+-------------+---------+-----------------+-------------+------------+-------------------+-----------------+



7.Join shipments with the warehouse master and calculate the number of shipments originating
from each region.

In [15]:
shipments_df.join(warehouse_df, "warehouse_id").groupBy("region").count().orderBy(F.desc("count")).show()

+------+-----+
|region|count|
+------+-----+
| South|33018|
|  West|26909|
| North| 9767|
|  East| 5306|
+------+-----+



8.Join shipments, carriers, and routes. Calculate average shipping cost per kilometre for each
carrier, excluding missing costs and zero distances.

In [17]:
shipments_df.join(carrier_df, "carrier_id") \
    .join(route_df, "route_id") \
    .filter(F.col("shipping_cost_inr").isNotNull() & (F.col("distance_km") > 0)) \
    .withColumn("cost_per_km", F.col("shipping_cost_inr") / F.col("distance_km")) \
    .groupBy("carrier_name") \
    .agg(
        F.round(F.avg("cost_per_km"), 2).alias("avg_cost_per_km"),
        F.count("*").alias("shipments_used")
    ) \
    .orderBy(F.desc("avg_cost_per_km")) \
    .show()

+--------------------+---------------+--------------+
|        carrier_name|avg_cost_per_km|shipments_used|
+--------------------+---------------+--------------+
|  AirSprint Couriers|           0.61|          7994|
|   BluePeak Delivery|           0.61|          8810|
|        ParcelBridge|            0.6|         14652|
|   SwiftLine Express|            0.6|         13621|
|MetroRoute Logistics|           0.59|         11581|
|           CargoNest|           0.59|          8671|
|           RouteWise|           0.59|          7310|
+--------------------+---------------+--------------+



**PART C | Null Handling and Conditional Logic**

9.Count missing values in carrier_id, package_weight_kg, shipping_cost_inr, and customer_rating.

In [18]:
shipments_df.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in ["carrier_id", "package_weight_kg", "shipping_cost_inr", "customer_rating"]]
).show()

+----------+-----------------+-----------------+---------------+
|carrier_id|package_weight_kg|shipping_cost_inr|customer_rating|
+----------+-----------------+-----------------+---------------+
|       871|             1129|             1503|          26255|
+----------+-----------------+-----------------+---------------+



10.Replace null handling_minutes values with zero in one DataFrame. Create another DataFrame
that excludes shipments with a missing carrier_id.

In [19]:
handling_filled_df = shipments_df.na.fill({"handling_minutes": 0})
has_carrier_df = shipments_df.na.drop(subset=["carrier_id"])

print("Null handling_minutes after fill:", handling_filled_df.filter(F.col("handling_minutes").isNull()).count())
print("Rows before / after dropping missing carrier:", shipments_df.count(), "/", has_carrier_df.count())

Null handling_minutes after fill: 0
Rows before / after dropping missing carrier: 75000 / 74129


11.Create weight_category using when() and otherwise(): Light (< 2 kg), Medium (2-10 kg), Heavy
(> 10 kg), and Unknown (null).

In [20]:
weighted_df = shipments_df.withColumn("weight_category",
                                  F.when(F.col("package_weight_kg").isNull(), "Unknown")
                                  .when(F.col("package_weight_kg") < 2, "Light")
                                  .when(F.col("package_weight_kg") <= 10, "Medium")
                                  .otherwise("Heavy"))
weighted_df.groupBy("weight_category").count().show()

+---------------+-----+
|weight_category|count|
+---------------+-----+
|        Unknown| 1129|
|         Medium|48577|
|          Light|22670|
|          Heavy| 2624|
+---------------+-----+



**PART D | Date and Time Functions**

12.Convert delivery-related date columns to Spark date types and calculate delivery duration in
days for delivered shipments.

In [25]:
date_cols = ["booking_date", "delivery_date", "actual_delivery_date"]
DATE_FMT = None  # Or specify a date format string if needed

def to_typed(df):
  for c in date_cols:
      if c in df.columns:
          df = df.withColumn(c, F.to_date(F.col(c), DATE_FMT) if DATE_FMT else F.to_date(F.col(c), "yyyy-MM-dd HH:mm:ss"))
  return df

shipments_df = to_typed(shipments_df)
shipments_df.select([c for c in date_cols if c in shipments_df.columns]).printSchema()

shipments_df.filter(F.col("shipment_status") == "Delivered") \
    .withColumn("delivery_duration", F.datediff(F.col("actual_delivery_date"), F.col("booking_date"))) \
    .select("shipment_id", "booking_date", "actual_delivery_date", "delivery_duration") \
    .show(10)

root
 |-- booking_date: date (nullable = true)
 |-- actual_delivery_date: date (nullable = true)

+-----------+------------+--------------------+-----------------+
|shipment_id|booking_date|actual_delivery_date|delivery_duration|
+-----------+------------+--------------------+-----------------+
| SHP0000001|  2026-05-24|          2026-05-27|                3|
| SHP0000002|  2026-05-17|          2026-05-21|                4|
| SHP0000003|  2026-01-31|          2026-02-04|                4|
| SHP0000004|  2026-03-12|          2026-03-18|                6|
| SHP0000005|  2026-08-16|          2026-08-29|               13|
| SHP0000006|  2026-02-06|          2026-02-09|                3|
| SHP0000009|  2026-08-12|          2026-08-21|                9|
| SHP0000010|  2026-06-01|          2026-06-11|               10|
| SHP0000011|  2026-04-24|          2026-05-01|                7|
| SHP0000013|  2026-06-26|          2026-06-29|                3|
+-----------+------------+------------------

13.Create delivery_performance as Early, On Time, or Late by comparing actual and promised
delivery dates. Give missing actual dates a separate classification.

In [30]:
performance_df = shipments_df.withColumn("delivery_performance",
                                         F.when(F.col("actual_delivery_date").isNull(), "Missing")
                                         .when(F.col("actual_delivery_date") < F.col("promised_delivery_date"), "Early")
                                         .when(F.col("actual_delivery_date") == F.col("promised_delivery_date"), "On Time")
                                         .otherwise("Late"))
performance_df.groupBy("delivery_performance").count().orderBy(F.desc("count")).show()

+--------------------+-----+
|delivery_performance|count|
+--------------------+-----+
|             Missing|22566|
|             On Time|22049|
|                Late|21900|
|               Early| 8485|
+--------------------+-----+



14.Using booking_date, calculate monthly delivered-shipment counts and total shipping fees.

In [32]:
shipments_df.filter(F.col("shipment_status") == "Delivered") \
    .withColumn("booking_month", F.month(F.col("booking_date"))) \
    .groupBy("Booking_month") \
    .agg(
        F.count("*").alias("delivered_shipments"),
        F.round(F.sum("shipping_fee_inr"), 2).alias("total_shipping_fees")
    ) \
    .orderBy(F.desc("booking_month")) \
    .show()

+-------------+-------------------+-------------------+
|Booking_month|delivered_shipments|total_shipping_fees|
+-------------+-------------------+-------------------+
|            8|               6192|         2306432.52|
|            7|               6773|         2512355.22|
|            6|               6615|         2447590.12|
|            5|               6858|          2549684.0|
|            4|               6379|         2389163.46|
|            3|               6810|         2530456.24|
|            2|               6097|          2268113.5|
|            1|               6710|         2525138.44|
+-------------+-------------------+-------------------+



**PART E | Window Functions**

15.Rank carriers by total shipping fees within each warehouse region using dense_rank().

In [33]:
carrier_fees = shipments_df.join(warehouse_df, "warehouse_id") \
    .groupBy(F.col("region"), "carrier_id") \
    .agg(F.round(F.sum("shipping_fee_inr"), 2).alias("total_fees"))
w15 = Window.partitionBy("region").orderBy(F.desc("total_fees"))
carrier_fees.withColumn("carrier_rank", F.dense_rank().over(w15)).orderBy("region", "carrier_rank").show()

+------+----------+----------+------------+
|region|carrier_id|total_fees|carrier_rank|
+------+----------+----------+------------+
|  East|       C01| 393862.62|           1|
|  East|       C02| 373590.02|           2|
|  East|       C03| 302260.92|           3|
|  East|       C05| 243322.92|           4|
|  East|       C07| 222025.86|           5|
|  East|       C04| 211583.64|           6|
|  East|       C06| 195810.34|           7|
|  East|      NULL|   24594.6|           8|
| North|       C02|  731442.1|           1|
| North|       C01| 653223.62|           2|
| North|       C03| 597883.46|           3|
| North|       C05| 464601.86|           4|
| North|       C07|  425396.7|           5|
| North|       C04| 393181.08|           6|
| North|       C06| 338296.46|           7|
| North|      NULL|   45235.7|           8|
| South|       C02|2442652.54|           1|
| South|       C01|2271284.54|           2|
| South|       C03|1929008.12|           3|
| South|       C07| 1519441.5|  

16.Identify the latest booked shipment for each warehouse using row_number().

In [34]:
w16 = Window.partitionBy("warehouse_id").orderBy(F.desc("booking_date"), F.desc("shipment_id"))
shipments_df.withColumn("row_num", F.row_number().over(w16)).filter(F.col("row_num") == 1).drop("row_num").select("warehouse_id", "shipment_id", "booking_date", "shipping_fee_inr").orderBy("warehouse_id").show()

+------------+-----------+------------+----------------+
|warehouse_id|shipment_id|booking_date|shipping_fee_inr|
+------------+-----------+------------+----------------+
|         W01| SHP0072346|  2026-08-28|          301.48|
|         W02| SHP0074049|  2026-08-28|          283.42|
|         W03| SHP0073860|  2026-08-28|          557.48|
|         W04| SHP0073268|  2026-08-28|           701.8|
|         W05| SHP0074305|  2026-08-28|          441.92|
|         W06| SHP0074983|  2026-08-28|          353.82|
|         W07| SHP0071003|  2026-08-28|          240.08|
|         W08| SHP0072690|  2026-08-28|          534.54|
+------------+-----------+------------+----------------+



17.Calculate a running total of shipping fees for each warehouse, ordered by booking date and
shipment ID.

In [35]:
w17 = Window.partitionBy("warehouse_id").orderBy("booking_date", "shipment_id")
shipments_df.withColumn("running_total", F.sum("shipping_fee_inr").over(w17)).orderBy("warehouse_id", "booking_date", "shipment_id").select("warehouse_id", "booking_date", "shipment_id", "shipping_fee_inr", "running_total").show(15)

+------------+------------+-----------+----------------+------------------+
|warehouse_id|booking_date|shipment_id|shipping_fee_inr|     running_total|
+------------+------------+-----------+----------------+------------------+
|         W01|  2026-01-01| SHP0004062|          312.48|            312.48|
|         W01|  2026-01-01| SHP0006739|          227.02|             539.5|
|         W01|  2026-01-01| SHP0007553|          316.02|            855.52|
|         W01|  2026-01-01| SHP0008505|          572.68|1428.1999999999998|
|         W01|  2026-01-01| SHP0009674|           322.9|            1751.1|
|         W01|  2026-01-01| SHP0011312|           390.1|            2141.2|
|         W01|  2026-01-01| SHP0011588|          421.72|           2562.92|
|         W01|  2026-01-01| SHP0011698|          190.22|           2753.14|
|         W01|  2026-01-01| SHP0011973|          384.64|3137.7799999999997|
|         W01|  2026-01-01| SHP0017305|          203.48|3341.2599999999998|
|         W0

**PART F | Union and Complex JSON**

18.Combine shipments_df and extra_df using unionByName(). Count the combined records.

In [36]:
extra_typed = to_typed(shipments_extra_df)
combined_df = shipments_df.unionByName(extra_typed)
print("shipments_df:", shipments_df.count(), "| shipments_extra_df:", shipments_extra_df.count(), "| Combined records:", combined_df.count())

shipments_df: 75000 | shipments_extra_df: 12000 | Combined records: 87000


19.Identify repeated shipment_id values and remove duplicate shipment records. Compare row
counts before and after deduplication.

In [37]:
dups = combined_df.groupBy("shipment_id").count().filter(F.col("count") > 1)
print("Shipment IDs that appear more than once:", dups.count())
dups.orderBy(F.desc("count")).show(5)

dedup_df = combined_df.dropDuplicates(["shipment_id"])
print("Rows before dedup:", combined_df.count(), "| Rows after dedup:", dedup_df.count(), "| Rows removed:", combined_df.count() - dedup_df.count())

Shipment IDs that appear more than once: 750
+-----------+-----+
|shipment_id|count|
+-----------+-----+
| SHP0039848|    2|
| SHP0048411|    2|
| SHP0052790|    2|
| SHP0026413|    2|
| SHP0041611|    2|
+-----------+-----+
only showing top 5 rows
Rows before dedup: 87000 | Rows after dedup: 86250 | Rows removed: 750


20.From the nested sensor JSON, extract warehouse ID, temperature limits, and sensor details. Use
explode() to produce one row per sensor.

In [38]:
cfg = spark.read.option("multiLine", True).json(BASE + "warehouse_sensor_config.json")
cfg.printSchema()

def array_cols(df):
    return [f.name for f in df.schema.fields if isinstance(f.dataType, ArrayType)]

def flatten_structs(df):
    """Expand struct columns into parent_child columns until none are left."""
    while any(isinstance(f.dataType, StructType) for f in df.schema.fields):
        cols = []
        for f in df.schema.fields:
            if isinstance(f.dataType, StructType):
                cols += [F.col(f"{f.name}.{s.name}").alias(f"{f.name}_{s.name}") for s in f.dataType.fields]
            else:
                cols.append(F.col(f.name))
        df = df.select(cols)
    return df

# If the file is wrapped like {"warehouses": [ {...}, {...} ]}, unwrap it first
if "warehouse_id" not in cfg.columns and len(array_cols(cfg)) == 1:
    wrap = array_cols(cfg)[0]
    cfg = cfg.select(F.explode(wrap).alias("w")).select("w.*")

flat = flatten_structs(cfg)
sensor_arrays = array_cols(flat)
print("Array column(s) found:", sensor_arrays)

sensors_df = flat.withColumn("sensor", F.explode(sensor_arrays[0])).drop(sensor_arrays[0])
sensors_df = flatten_structs(sensors_df)
print("One row per sensor:", sensors_df.count())
sensors_df.show(20, truncate=False)

root
 |-- environment: struct (nullable = true)
 |    |-- humidity_target_pct: long (nullable = true)
 |    |-- temperature_range_c: struct (nullable = true)
 |    |    |-- maximum: long (nullable = true)
 |    |    |-- minimum: long (nullable = true)
 |-- safety: struct (nullable = true)
 |    |-- fire_suppression: string (nullable = true)
 |    |-- inspections: struct (nullable = true)
 |    |    |-- last_date: string (nullable = true)
 |    |    |-- next_date: string (nullable = true)
 |-- sensors: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- enabled: boolean (nullable = true)
 |    |    |-- sensor_id: string (nullable = true)
 |    |    |-- type: string (nullable = true)
 |-- warehouse_id: string (nullable = true)

Array column(s) found: ['sensors']
One row per sensor: 24
+-------------------------------+---------------------------------------+---------------------------------------+-----------------------+----------------------------+-----

**PART G | Lazy Execution, Caching, and Persistence**

21.Build a pipeline using filter(), join(), withColumn(), groupBy(), and orderBy(). Inspect its plan with
explain() before calling an action.

In [39]:
pipeline = (shipments_df
    .filter(F.col("shipment_status") == "Delivered")
    .join(warehouse_df, "warehouse_id")
    .withColumn("delivery_days", F.datediff(F.col("actual_delivery_date"), F.col("booking_date")))
    .groupBy("region")
    .agg(F.count("*").alias("delivered"),
         F.round(F.avg("delivery_days"), 2).alias("avg_delivery_days"),
         F.round(F.sum("shipping_fee_inr"), 2).alias("total_fees"))
    .orderBy(F.desc("total_fees")))

print("No job has run yet. Plan:")
pipeline.explain(True)
print("Triggering the action now:")
pipeline.show()

No job has run yet. Plan:
== Parsed Logical Plan ==
'Sort ['total_fees DESC NULLS LAST], true
+- Aggregate [region#94], [region#94, count(1) AS delivered#1700L, round(avg(delivery_days#1699), 2) AS avg_delivery_days#1701, round(sum(shipping_fee_inr#28), 2) AS total_fees#1702]
   +- Project [warehouse_id#22, shipment_id#17, booking_date#994, dispatch_date#19, promised_delivery_date#20, actual_delivery_date#995, carrier_id#23, route_id#24, customer_segment#25, service_level#26, package_weight_kg#27, shipping_fee_inr#28, shipping_cost_inr#29, shipment_status#30, delivery_attempts#31, damage_flag#32, customer_rating#33, handling_minutes#34, payment_mode#35, priority_flag#36, warehouse_name#92, city#93, region#94, capacity_units#95, storage_type#96, ... 1 more fields]
      +- Project [warehouse_id#22, shipment_id#17, booking_date#994, dispatch_date#19, promised_delivery_date#20, actual_delivery_date#995, carrier_id#23, route_id#24, customer_segment#25, service_level#26, package_weight_kg#2

22.Cache delivered shipments. Compare execution times for repeated count() and aggregation
actions.

In [42]:
import time

delivered_shipments_df = shipments_df.filter(F.col("shipment_status") == "Delivered")
delivered_shipments_df.cache()

def time_execution(label, func):
    start = time.time()
    func()
    end = time.time()
    print(f"{label}: {end - start:.4f} seconds")

print("--- Cached DataFrame ---")
time_execution("count #1 (fills cache)", lambda: delivered_shipments_df.count())
time_execution("count #2 (from cache)", lambda: delivered_shipments_df.count())

agg = lambda d: d.groupBy("service_level").agg(F.sum("shipping_fee_inr").alias("fees")).collect()

time_execution("agg #1", lambda: agg(delivered_shipments_df))
time_execution("agg #2", lambda: agg(delivered_shipments_df))

print("--- Uncached DataFrame ---")
time_execution("count (no cache)", lambda: delivered_shipments_df.count())
time_execution("agg (no cache)", lambda: agg(delivered_shipments_df))

delivered_shipments_df.unpersist()

--- Cached DataFrame ---
count #1 (fills cache): 3.0795 seconds
count #2 (from cache): 0.3837 seconds
agg #1: 0.8011 seconds
agg #2: 0.5259 seconds
--- Uncached DataFrame ---
count (no cache): 0.2462 seconds
agg (no cache): 0.4405 seconds


DataFrame[shipment_id: string, booking_date: date, dispatch_date: date, promised_delivery_date: date, actual_delivery_date: date, warehouse_id: string, carrier_id: string, route_id: string, customer_segment: string, service_level: string, package_weight_kg: double, shipping_fee_inr: double, shipping_cost_inr: double, shipment_status: string, delivery_attempts: int, damage_flag: string, customer_rating: double, handling_minutes: double, payment_mode: string, priority_flag: string]

23.Persist Express shipments using MEMORY_AND_DISK. Run two actions, inspect the storage level,
and release the data using unpersist().

In [43]:
express_df = shipments_df.filter(F.col("service_level") == "Express").persist(StorageLevel.MEMORY_AND_DISK)

print("Action 1 - count:", express_df.count())
print("Action 2 - avg fee:", express_df.agg(F.avg("shipping_fee_inr")).first()[0])
print("Storage level:", express_df.storageLevel)

express_df.unpersist()
print("Cached after unpersist:", express_df.is_cached)

Action 1 - count: 21090
Action 2 - avg fee: 476.93068089141707
Storage level: Disk Memory Serialized 1x Replicated
Cached after unpersist: False


**PART H | Partitioning and Optimization**

24.Repartition shipments into 8 partitions. Use spark_partition_id() to count records in each
partition.

In [44]:
print("Original partitions:", shipments_df.rdd.getNumPartitions())
shipments_df8 = shipments_df.repartition(8)
(shipments_df8.withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id").count().orderBy("partition_id").show())

Original partitions: 2
+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 9375|
|           1| 9374|
|           2| 9376|
|           3| 9375|
|           4| 9375|
|           5| 9375|
|           6| 9375|
|           7| 9375|
+------------+-----+



25.Repartition shipments into 12 partitions, then reduce to 3 using coalesce(). Compare with
repartition(3) using execution plans.

In [45]:
shipments_df12 = shipments_df.repartition(12)
print("Partitions:", shipments_df12.rdd.getNumPartitions())

print("=== coalesce(3) ===")
shipments_df12.coalesce(3).explain()
print("=== repartition(3) ===")
shipments_df12.repartition(3).explain()

Partitions: 12
=== coalesce(3) ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Coalesce 3
   +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=3158]
      +- Project [shipment_id#17, cast(gettimestamp(booking_date#18, yyyy-MM-dd HH:mm:ss, TimestampType, try_to_timestamp, Some(Etc/UTC), true) as date) AS booking_date#994, dispatch_date#19, promised_delivery_date#20, cast(gettimestamp(actual_delivery_date#21, yyyy-MM-dd HH:mm:ss, TimestampType, try_to_timestamp, Some(Etc/UTC), true) as date) AS actual_delivery_date#995, warehouse_id#22, carrier_id#23, route_id#24, customer_segment#25, service_level#26, package_weight_kg#27, shipping_fee_inr#28, shipping_cost_inr#29, shipment_status#30, delivery_attempts#31, damage_flag#32, customer_rating#33, handling_minutes#34, payment_mode#35, priority_flag#36]
         +- FileScan csv [shipment_id#17,booking_date#18,dispatch_date#19,promised_delivery_date#20,actual_delivery_date#21,warehouse_id#22,carrier_id#23,route

26.Join shipment and warehouse data, then write the result as Parquet using 4 output partitions.
Inspect the generated files in Colab.

In [46]:
shipments_warehouse_df = shipments_df.join(warehouse_df, "warehouse_id", "left")
output = "/content/output/shipments_warehouse_parquet"
shipments_warehouse_df.repartition(4).write.mode("overwrite").parquet(output)

for root, _, files_ in os.walk(output):
    for f in sorted(files_):
        p = os.path.join(root, f)
        print(f"{os.path.getsize(p):>10,} bytes  {p}")
print("Parquet part files:", len(glob.glob(output + "/part-*.parquet")))

         8 bytes  /content/output/shipments_warehouse_parquet/._SUCCESS.crc
     3,732 bytes  /content/output/shipments_warehouse_parquet/.part-00000-8cadd762-77dd-4fdf-be87-e602a0fc8d8f-c000.snappy.parquet.crc
     3,736 bytes  /content/output/shipments_warehouse_parquet/.part-00001-8cadd762-77dd-4fdf-be87-e602a0fc8d8f-c000.snappy.parquet.crc
     3,728 bytes  /content/output/shipments_warehouse_parquet/.part-00002-8cadd762-77dd-4fdf-be87-e602a0fc8d8f-c000.snappy.parquet.crc
     3,736 bytes  /content/output/shipments_warehouse_parquet/.part-00003-8cadd762-77dd-4fdf-be87-e602a0fc8d8f-c000.snappy.parquet.crc
         0 bytes  /content/output/shipments_warehouse_parquet/_SUCCESS
   476,215 bytes  /content/output/shipments_warehouse_parquet/part-00000-8cadd762-77dd-4fdf-be87-e602a0fc8d8f-c000.snappy.parquet
   476,783 bytes  /content/output/shipments_warehouse_parquet/part-00001-8cadd762-77dd-4fdf-be87-e602a0fc8d8f-c000.snappy.parquet
   476,126 bytes  /content/output/shipments_warehouse

**FINAL CHALLENGE | Logistics Performance Pipeline**

Build one end-to-end PySpark pipeline that joins shipment, warehouse, carrier, and route data.
Complete all requirements below:
1. Clean missing shipping costs and package weights using a clearly explained strategy.
2. Calculate delivery duration, late-delivery indicators, and shipment profit.
3. Aggregate revenue, profit, and late-delivery percentage by region and carrier.
4. Rank carriers within each region by on-time delivery percentage.
5. Cache the final aggregated DataFrame and reuse it for multiple actions.
6. Save the final report as partitioned Parquet files.

In [47]:
# 1. Clean missing shipping cost / weight
svc_stats = (dedup_df.groupBy("service_level").agg(
        F.avg("shipping_cost_inr").alias("svc_avg_cost"),
        F.percentile_approx("package_weight_kg", 0.5).alias("svc_median_weight")))

g = dedup_df.agg(F.avg("shipping_cost_inr").alias("g_cost"),
                 F.percentile_approx("package_weight_kg", 0.5).alias("g_weight")).first()

clean_df = (dedup_df.join(svc_stats, "service_level", "left")
    .withColumn("cost_imputed", F.col("shipping_cost_inr").isNull())
    .withColumn("weight_imputed", F.col("package_weight_kg").isNull())
    .withColumn("shipping_cost_clean", F.coalesce(F.col("shipping_cost_inr"), F.col("svc_avg_cost"), F.lit(g["g_cost"])))
    .withColumn("package_weight_clean", F.coalesce(F.col("package_weight_kg"), F.col("svc_median_weight"), F.lit(g["g_weight"]))))

# Join warehouse, carrier and route data
enriched_df = (clean_df
    .filter((F.col("shipment_status") == "Delivered") & F.col("actual_delivery_date").isNotNull())
    .join(warehouse_df, "warehouse_id", "left")
    .join(carrier_df, "carrier_id", "left")
    .join(route_df, "route_id", "left"))

# 2. Calculate Delivery duration, late-delivery indicator, shipment profit
enriched_df = (enriched_df
    .withColumn("delivery_days", F.datediff(F.col("actual_delivery_date"), F.col("booking_date")))
    .withColumn("is_late", F.when(F.col("actual_delivery_date") > F.col("promised_delivery_date"), 1).otherwise(0))
    .withColumn("shipment_profit", F.col("shipping_fee_inr") - F.col("shipping_cost_clean"))
    .withColumn("cost_per_km", F.when(F.col("distance_km") > 0, F.col("shipping_cost_clean") / F.col("distance_km"))))

# 3. Aggregate revenue, profit and late-delivery percentage by region and carrier
report_df = (enriched_df.groupBy("region", "carrier_id", "carrier_name").agg(
        F.count("*").alias("delivered_shipments"),
        F.round(F.sum("shipping_fee_inr"), 2).alias("total_revenue"),
        F.round(F.sum("shipment_profit"), 2).alias("total_profit"),
        F.round(F.avg("delivery_days"), 2).alias("avg_delivery_days"),
        F.round(F.avg("is_late") * 100, 2).alias("late_delivery_pct"))
    .withColumn("on_time_pct", F.round(100 - F.col("late_delivery_pct"), 2)))

# 4. Rank carriers within each region by on-time delivery %
w_rank = Window.partitionBy("region").orderBy(F.desc("on_time_pct"))
report_df = report_df.withColumn("carrier_rank", F.dense_rank().over(w_rank))

# 5. Cache the final aggregated DataFrame
report_df = report_df.cache()

print("Execution plan (relevant for submission):")
report_df.explain()

Execution plan (relevant for submission):
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [region#94, carrier_id#23, carrier_name#115, delivered_shipments#6490L, total_revenue#6491, total_profit#6492, avg_delivery_days#6493, late_delivery_pct#6494, on_time_pct#6544, carrier_rank#6545]
      +- InMemoryRelation [region#94, carrier_id#23, carrier_name#115, delivered_shipments#6490L, total_revenue#6491, total_profit#6492, avg_delivery_days#6493, late_delivery_pct#6494, on_time_pct#6544, carrier_rank#6545], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- AdaptiveSparkPlan isFinalPlan=false
               +- Window [dense_rank(on_time_pct#6544) windowspecdefinition(region#94, on_time_pct#6544 DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS carrier_rank#6545], [region#94], [on_time_pct#6544 DESC NULLS LAST]
                  +- Sort [region#94 ASC NULLS FIRST, on_time_pct#6544 DESC NULLS LAST], false, 

**Reuse the cached report for multiple actions**

In [49]:
print("Action 1: full report")
report_df.orderBy("region", "carrier_rank").show()

print("Action 2: best carrier per region")
report_df.filter(F.col("carrier_rank") == 1).orderBy("region").show()

print("Action 3: region summary")
(report_df.groupBy("region").agg(
    F.sum("delivered_shipments").alias("shipments"),
    F.round(F.sum("total_revenue"), 2).alias("revenue"),
    F.round(F.sum("total_profit"), 2).alias("profit"))
 .orderBy(F.desc("revenue")).show())

print("Action 4: rows in report:", report_df.count())
report_df.explain()   # now shows InMemoryTableScan

Action 1: full report
+------+----------+--------------------+-------------------+-------------+------------+-----------------+-----------------+-----------+------------+
|region|carrier_id|        carrier_name|delivered_shipments|total_revenue|total_profit|avg_delivery_days|late_delivery_pct|on_time_pct|carrier_rank|
+------+----------+--------------------+-------------------+-------------+------------+-----------------+-----------------+-----------+------------+
|  East|       C01|   SwiftLine Express|                837|    319401.02|    76147.64|             6.43|            38.47|      61.53|           1|
|  East|       C04|  AirSprint Couriers|                435|    163889.12|    40399.28|             6.53|             40.0|       60.0|           2|
|  East|       C05|           CargoNest|                539|    198205.82|    47167.47|             6.76|            41.74|      58.26|           3|
|  East|       C07|   BluePeak Delivery|                491|     176433.4|     39233

In [50]:
# 6. Save the final report as partitioned Parquet files.
output_final = "/content/output/logistics_report_parquet"
(report_df.repartition(F.col("region"))
    .write.mode("overwrite")
    .partitionBy("region")
    .parquet(output_final))

for root, _, files_ in os.walk(output_final):
    for f in sorted(files_):
        if f.endswith(".parquet"):
            print(os.path.join(root, f))

# Verify by reading back
spark.read.parquet(output_final).orderBy("region", "carrier_rank").show(10)

/content/output/logistics_report_parquet/region=West/part-00164-28f05baf-2fdb-4ac6-8d1b-0cfca45aa26a.c000.snappy.parquet
/content/output/logistics_report_parquet/region=North/part-00192-28f05baf-2fdb-4ac6-8d1b-0cfca45aa26a.c000.snappy.parquet
/content/output/logistics_report_parquet/region=South/part-00090-28f05baf-2fdb-4ac6-8d1b-0cfca45aa26a.c000.snappy.parquet
/content/output/logistics_report_parquet/region=East/part-00129-28f05baf-2fdb-4ac6-8d1b-0cfca45aa26a.c000.snappy.parquet
+----------+--------------------+-------------------+-------------+------------+-----------------+-----------------+-----------+------------+------+
|carrier_id|        carrier_name|delivered_shipments|total_revenue|total_profit|avg_delivery_days|late_delivery_pct|on_time_pct|carrier_rank|region|
+----------+--------------------+-------------------+-------------+------------+-----------------+-----------------+-----------+------------+------+
|       C01|   SwiftLine Express|                837|    319401.02|